# SpendDNA — Your Wallet's Year-End Story
**Name:** Mohamed  |  **Batch:** <your batch>  |  **Date:** <submission date>

**The Unlox Academy — Week 2 Minor Project**
Analysis of 6 months of synthetic UPI/bank transactions (Rahul Sharma, Jan–Jun 2024).
Tools used: Python fundamentals, NumPy, Pandas only. No regex, no matplotlib, no ML.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

pd.set_option('display.width', 120)

# Upload the CSV in Google Colab if it is not already in the working folder.
csv_path = Path('rahul_transactions.csv')
if not csv_path.exists():
    try:
        from google.colab import files
        print("Upload rahul_transactions.csv to continue:")
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
        if not csv_files:
            raise FileNotFoundError("No CSV file was uploaded. Please upload the transaction CSV.")
        csv_path = Path(csv_files[0])
    except ImportError:
        raise FileNotFoundError(
            "CSV not found. Place rahul_transactions.csv in the same folder as this notebook."
        )

df_raw = pd.read_csv(csv_path)
required_columns = {'Date', 'Amount', 'Type', 'Time', 'Mode', 'Description'}
missing_columns = required_columns - set(df_raw.columns)
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {sorted(missing_columns)}")
print('Loaded file:', csv_path.name)
print('Raw shape:', df_raw.shape)
df_raw.head()


## Feature 1 — Transaction Parser

In [ ]:
# Clean transaction rows and standardise dates, amounts, types, and times.
df = df_raw.copy()

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
dupes_dropped = before - len(df)

def parse_date(value):
    if pd.isna(value):
        return pd.NaT
    value = str(value).strip()
    for fmt in ('%d/%m/%y', '%Y-%m-%d', '%d-%b-%y', '%d %b %Y'):
        try:
            return pd.Timestamp(datetime.strptime(value, fmt))
        except ValueError:
            continue
    return pd.NaT

df['date'] = df['Date'].apply(parse_date)
df['amount'] = (df['Amount'].astype(str)
                .str.replace('₹', '', regex=False)
                .str.replace('Rs.', '', regex=False)
                .str.replace(',', '', regex=False)
                .str.strip())
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')

type_map = {'dr': 'debit', 'debit': 'debit', 'cr': 'credit', 'credit': 'credit'}
df['type'] = df['Type'].astype(str).str.strip().str.lower().map(type_map)
df['Mode'] = df['Mode'].replace('', np.nan)

# Extract the hour safely from common HH:MM or HH:MM:SS time strings.
time_text = df['Time'].astype(str).str.strip()
df['hour'] = pd.to_numeric(time_text.str.extract(r'^(\d{1,2})', expand=False), errors='coerce')
df.loc[~df['hour'].between(0, 23), 'hour'] = np.nan
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.day_name()

# Exclude rows that cannot be used for numeric/time-based analysis.
invalid_core = df['date'].isna() | df['amount'].isna() | df['type'].isna()
print(f"Parsed {len(df)} rows; dropped {dupes_dropped} exact duplicates.")
print(f"Rows with invalid date/amount/type: {int(invalid_core.sum())}")
print(df[['date', 'amount', 'type', 'hour']].dtypes)


## Feature 2 — Vendor Extractor

In [ ]:
# Step 1: inspect the messy descriptions
print(df['Description'].nunique(), 'unique descriptions')
print(df['Description'].unique()[:25])

In [ ]:
# AI-assisted: general approach (keyword dictionary + first-match function).
# The keyword lists were built by inspecting the Description column of this dataset.
# ORDER MATTERS: the first vendor whose keyword matches wins, so specific vendors
# (Instamart, Amazon Prime, Salary...) come BEFORE general ones (Swiggy, Amazon).
vendor_keywords = {
    'Salary':            ['SALARY'],
    'Landlord Rent':     ['RENT-LANDLORD'],
    'Cash Withdrawal':   ['ATM-WDL'],
    'Swiggy Instamart':  ['INSTAMART'],
    'Swiggy':            ['SWIGGY', 'BUNDL'],          # BUNDL = Swiggy's parent company
    'Zomato':            ['ZOMATO'],
    'Zepto':             ['ZEPTO', 'KIRANAKART'],      # Kiranakart = Zepto's parent
    'Blinkit':           ['BLINKIT', 'GROFERS'],       # Grofers = Blinkit's old name
    'Amazon Prime':      ['AMZN PRIME', 'AMAZON PRIME', 'AMAZON-PRIME'],
    'Amazon':            ['AMAZON', 'AMZN'],
    'Flipkart':          ['FLIPKART', 'FKART'],
    'Myntra':            ['MYNTRA'],
    'Nykaa':             ['NYKAA', 'FSN'],             # FSN = Nykaa's parent
    'Uber':              ['UBER'],
    'Ola':               ['OLA CABS', 'OLACABS', 'OLA ELECTRIC', 'OLA-PRIME', 'ANI TECHNOLOGIES'],
    'Rapido':            ['RAPIDO', 'ROPPEN'],         # Roppen = Rapido's parent
    'BMTC':              ['BMTC', 'TUMMOC'],
    'Starbucks':         ['STARBUCKS'],
    'Third Wave Coffee': ['THIRD WAVE', 'THIRDWAVE', 'TWC'],
    'Cafe Coffee Day':   ['COFFEE DAY', 'CCD'],
    'Empire Restaurant': ['EMPIRE'],
    'Meghana Foods':     ['MEGHANA'],
    'Truffles':          ['TRUFFLES'],
    'Dineout':           ['DINEOUT'],
    'Local Restaurant':  ['RESTAURANT'],
    'DMart':             ['DMART', 'AVENUE SUPERMARTS'],
    'BigBasket':         ['BIGBASKET', 'INNOVATIVE RETAIL'],
    'Zerodha':           ['ZERODHA'],
    'Groww':             ['GROWW', 'NEXTBILLION'],
    'Indian Oil':        ['INDIAN OIL', 'IOC'],
    'HP Petrol':         ['HP PETROL'],
    'BPCL':              ['BPCL'],
    'BESCOM':            ['BESCOM', 'ELEC SUPPLY'],
    'BWSSB':             ['BWSSB'],
    'Airtel':            ['AIRTEL'],
    'Jio':               ['JIO'],
    'Vi':                ['VI POSTPAID', 'VI-RECHARGE', 'VODAFONE'],
    'Netflix':           ['NETFLIX'],
    'Spotify':           ['SPOTIFY'],
    'Hotstar':           ['HOTSTAR', 'STAR INDIA'],
    'BookMyShow':        ['BOOKMYSHOW', 'BMS', 'BIGTREE'],
}

def extract_vendor(description):
    text = description.upper()
    for vendor, keywords in vendor_keywords.items():
        for kw in keywords:
            if kw in text:
                return vendor
    # nothing matched: a UPI string with a person's name = friend-to-friend transfer
    if text.startswith('UPI-'):
        return 'P2P Transfer'
    return 'Uncategorised'

df['vendor_clean'] = df['Description'].apply(extract_vendor)

print('Unique vendors:', df['vendor_clean'].nunique())
print(df['vendor_clean'].value_counts().head(10))

## Feature 3 — Category Tagger

In [ ]:
category_map = {
    'Swiggy': 'Food Delivery', 'Zomato': 'Food Delivery',
    'Swiggy Instamart': 'Quick Commerce', 'Zepto': 'Quick Commerce', 'Blinkit': 'Quick Commerce',
    'Amazon': 'E-commerce', 'Flipkart': 'E-commerce', 'Myntra': 'E-commerce', 'Nykaa': 'E-commerce',
    'Uber': 'Transport', 'Ola': 'Transport', 'Rapido': 'Transport', 'BMTC': 'Transport',
    'Starbucks': 'Cafe', 'Third Wave Coffee': 'Cafe', 'Cafe Coffee Day': 'Cafe',
    'Empire Restaurant': 'Restaurants', 'Meghana Foods': 'Restaurants', 'Truffles': 'Restaurants',
    'Dineout': 'Restaurants', 'Local Restaurant': 'Restaurants',
    'Netflix': 'Subscriptions', 'Spotify': 'Subscriptions', 'Hotstar': 'Subscriptions',
    'Amazon Prime': 'Subscriptions',
    'BESCOM': 'Utilities', 'BWSSB': 'Utilities', 'Airtel': 'Utilities', 'Jio': 'Utilities',
    'Vi': 'Utilities', 'Landlord Rent': 'Utilities',   # rent grouped under Utilities (my choice)
    'DMart': 'Groceries', 'BigBasket': 'Groceries',
    'Zerodha': 'Investments', 'Groww': 'Investments',
    'Indian Oil': 'Fuel', 'HP Petrol': 'Fuel', 'BPCL': 'Fuel',
    'BookMyShow': 'Entertainment',
    'P2P Transfer': 'Personal Transfer',
    'Cash Withdrawal': 'Cash Withdrawal',
    'Salary': 'Income',
}
df['category'] = df['vendor_clean'].map(category_map).fillna('Uncategorised')
print(df['category'].value_counts())

## Feature 4 — Spending Overview

In [ ]:
debits = df[df['type'] == 'debit'].copy()
credits = df[df['type'] == 'credit'].copy()

total_credit = credits['amount'].sum()
total_debit = debits['amount'].sum()
net = total_credit - total_debit
savings_rate = (net / total_credit * 100) if total_credit else 0.0

cat_spend = debits.groupby('category')['amount'].sum().sort_values(ascending=False)
vendor_spend = debits.groupby('vendor_clean')['amount'].agg(['sum', 'count']).sort_values('sum', ascending=False)

print(f'Total credits : Rs. {total_credit:,.0f}')
print(f'Total debits  : Rs. {total_debit:,.0f}')
print(f'Net change    : Rs. {net:,.0f}')
print(f'Savings rate  : {savings_rate:.1f}%')
print('Top 5 categories:'); print(cat_spend.head(5))
print('Top 5 vendors:'); print(vendor_spend.head(5))


## Feature 5 — Monthly Trend Analysis

In [ ]:
month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun'}
month_pivot = debits.pivot_table(values='amount', index='category', columns='month',
                                 aggfunc='sum', fill_value=0)
# Keep all six months even if a category/month has no transactions.
month_pivot = month_pivot.reindex(columns=range(1, 7), fill_value=0)
month_pivot.columns = [month_names[m] for m in month_pivot.columns]

first = month_pivot['Jan'].to_numpy(dtype=float)
last = month_pivot['Jun'].to_numpy(dtype=float)
pct_change = np.where(first > 0, (last - first) / np.where(first > 0, first, 1) * 100, np.nan)
trend = pd.Series(pct_change, index=month_pivot.index).dropna().sort_values()

print(month_pivot.round(0).astype(int))
if not trend.empty:
    print('Biggest growth  :', trend.index[-1], f'{trend.iloc[-1]:+.0f}%')
    print('Biggest decline :', trend.index[0], f'{trend.iloc[0]:+.0f}%')
else:
    print('Monthly change unavailable: no category had spending in January.')


## Feature 6 — Time-of-Day Patterns

In [ ]:
# Category x hour matrix (NumPy array: categories x 24)
cats = list(cat_spend.index)
hour_matrix = np.zeros((len(cats), 24), dtype=int)
valid_hour_debits = debits[debits['hour'].notna()].copy()
for i, c in enumerate(cats):
    hours = valid_hour_debits.loc[valid_hour_debits['category'] == c, 'hour'].astype(int)
    for h in hours:
        hour_matrix[i, h] += 1

food = valid_hour_debits[valid_hour_debits['category'] == 'Food Delivery']
late_mask = (food['hour'] >= 21) | (food['hour'] < 2)
late_food_pct = late_mask.mean() * 100 if len(food) else 0.0
print(f'Food Delivery orders between 21:00 and 02:00: {late_food_pct:.1f}%')

cafe = valid_hour_debits[valid_hour_debits['category'] == 'Cafe']
morning_cafe_pct = cafe['hour'].between(8, 10).mean() * 100 if len(cafe) else 0.0
print(f'Cafe visits between 08:00 and 11:00: {morning_cafe_pct:.1f}%')

if 'Food Delivery' in cats:
    row = hour_matrix[cats.index('Food Delivery')]
    row_max = row.max()
    for h in range(24):
        bar = '#' * (row[h] * 40 // row_max) if row_max else ''
        print(f'{h:02d}:00 {bar:<40} {row[h]}')
else:
    print('No Food Delivery transactions were categorised.')


## Feature 7 — Anomaly Detection (z-score within each category)

In [ ]:
# Flag unusually large transactions relative to others in the same category.
spend = debits.copy()
cat_mean = spend.groupby('category')['amount'].transform('mean')
cat_std = spend.groupby('category')['amount'].transform('std').replace(0, np.nan)
spend['z_score'] = ((spend['amount'] - cat_mean) / cat_std).fillna(0)

anomalies = spend[spend['z_score'] > 2].sort_values('z_score', ascending=False)
print('Anomalies flagged:', len(anomalies))
anomalies[['date', 'vendor_clean', 'category', 'amount', 'z_score']].head(5)


## Feature 8 — Spending Archetype Detection

In [ ]:
# Rule-based spending archetypes, based on this dataset's calculated metrics.
def pct_of_debits(*cat_names):
    return (debits.loc[debits['category'].isin(cat_names), 'amount'].sum() / total_debit * 100) if total_debit else 0.0

def is_foodie():
    p = pct_of_debits('Food Delivery', 'Restaurants', 'Cafe')
    return p > 25, f'{p:.1f}% on food'
def is_quick_commerce_junkie():
    p = pct_of_debits('Quick Commerce')
    return p > 15, f'{p:.1f}% on quick commerce'
def is_shopaholic():
    p = pct_of_debits('E-commerce')
    return p > 15, f'{p:.1f}% on e-commerce'
def is_investor():
    p = pct_of_debits('Investments')
    return p > 15, f'{p:.1f}% on investments'
def is_late_night_snacker():
    return late_food_pct > 50, f'{late_food_pct:.0f}% food orders 9 PM–2 AM'
def is_cab_commuter():
    p = pct_of_debits('Transport')
    return p > 10, f'{p:.1f}% on transport'
def is_subscription_lover():
    n = debits.loc[debits['category'] == 'Subscriptions', 'vendor_clean'].nunique()
    return n >= 5, f'{n} subscription vendors'
def is_yolo_spender():
    return savings_rate < 10, f'savings rate {savings_rate:.0f}%'
def is_disciplined_saver():
    return savings_rate > 40, f'savings rate {savings_rate:.0f}%'

archetype_rules = {
    'THE FOODIE': is_foodie,
    'THE QUICK COMMERCE JUNKIE': is_quick_commerce_junkie,
    'THE SHOPAHOLIC': is_shopaholic,
    'THE INVESTOR': is_investor,
    'THE LATE-NIGHT SNACKER': is_late_night_snacker,
    'THE CAB COMMUTER': is_cab_commuter,
    'THE SUBSCRIPTION LOVER': is_subscription_lover,
    'THE YOLO SPENDER': is_yolo_spender,
    'THE DISCIPLINED SAVER': is_disciplined_saver,
}

matched = []
for name, rule in archetype_rules.items():
    ok, metric = rule()
    if ok:
        matched.append((name, metric))
        print(f'-> {name:<28} ({metric})')
if not matched:
    print('No archetype thresholds were met.')


## Bonus — Day-of-week analysis & vendor cleanup audit

In [ ]:
# Compare average spend on dates with transactions (not every calendar day).
spend_days = spend.copy()
spend_days['is_weekend'] = spend_days['day_of_week'].isin(['Saturday', 'Sunday'])
daily = spend_days.groupby(['date', 'is_weekend'])['amount'].sum().reset_index()
avg = daily.groupby('is_weekend')['amount'].mean()
weekday_avg = avg.get(False, np.nan)
weekend_avg = avg.get(True, np.nan)
if pd.notna(weekday_avg):
    print(f'Avg weekday spend/day: Rs. {weekday_avg:,.0f}')
else:
    print('No weekday spending dates found.')
if pd.notna(weekend_avg):
    comparison = (weekend_avg / weekday_avg - 1) * 100 if pd.notna(weekday_avg) and weekday_avg else np.nan
    comparison_text = f' ({comparison:+.0f}% vs weekday)' if pd.notna(comparison) else ''
    print(f'Avg weekend spend/day: Rs. {weekend_avg:,.0f}{comparison_text}')
else:
    print('No weekend spending dates found.')

unmapped = df[df['vendor_clean'] == 'Uncategorised']['Description'].dropna().unique()
print('Unmapped descriptions:', len(unmapped), list(unmapped))


## Final Printed Report

In [ ]:
def bar(value, max_value, width=20):
    if not max_value or pd.isna(max_value):
        return ''
    return '#' * int(round(value / max_value * width))

line = '=' * 64
print(line)
print('  SpendDNA REPORT - RAHUL SHARMA')
print(f"  {len(df):,} transactions - Jan to Jun 2024")
print(line)
print('\n  EXECUTIVE SUMMARY')
print(f'    Total credits   : Rs. {total_credit:>12,.0f}')
print(f'    Total debits    : Rs. {total_debit:>12,.0f}')
print(f'    Net change      : Rs. {net:>12,.0f}')
print(f'    Savings rate    : {savings_rate:>12.1f}%')
print(f'    Transactions    : {len(df):>12,}')
print(f"    Unique vendors  : {df['vendor_clean'].nunique():>12}")
print('\n  TOP CATEGORIES (% of debit total)')
top5 = cat_spend.head(5)
for c, v in top5.items():
    share = v / total_debit * 100 if total_debit else 0
    print(f'    {c:<16} {bar(v, top5.max()):<20} {share:5.1f}%  Rs. {v:>10,.0f}')
print('\n  TOP VENDORS')
for v, row in vendor_spend.head(5).iterrows():
    print(f"    {v:<18} Rs. {row['sum']:>10,.0f}   ({int(row['count']):>3} txns)")
print('\n  TIME-OF-DAY PATTERNS')
print(f'    Food Delivery : {late_food_pct:.0f}% of orders between 21:00 and 02:00')
print(f'    Cafe          : {morning_cafe_pct:.0f}% of visits between 08:00 and 11:00')
print('\n  MONTHLY TREND (Food Delivery)')
if 'Food Delivery' in month_pivot.index:
    fd = month_pivot.loc['Food Delivery']
    for m, v in fd.items():
        print(f'    {m} Rs. {v:>9,.0f} {bar(v, fd.max(), 25)}')
else:
    print('    No Food Delivery transactions found.')
print('\n  TOP ANOMALIES (z-score within category)')
for _, r in anomalies.head(5).iterrows():
    date_value = pd.to_datetime(r.get('date'), errors='coerce')
    date_text = date_value.strftime('%d %b') if pd.notna(date_value) else 'Unknown date'
    print(
        f"    {date_text} - {str(r.get('vendor_clean', 'Unknown')):<12} "
        f"Rs. {float(r.get('amount', 0)):>9,.0f}   "
        f"(z={float(r.get('z_score', 0)):.1f})"
    )
print("\n  RAHUL'S SPENDING ARCHETYPES")
for name, metric in matched:
    print(f'    -> {name:<26} ({metric})')
if not matched:
    print('    No archetype thresholds were met.')
print(line)


## Key Insights

Use the actual values from the report to complete these three data-specific points:

1. **Savings:** Total credits were Rs. ___ and total debits were Rs. ___, leaving a net change of Rs. ___ and a savings rate of ___%.
2. **Late-night food:** ___% of Food Delivery transactions occurred between 9 PM and 2 AM, showing the timing pattern in the transaction data.
3. **Spending concentration:** The largest spending category was ___ at Rs. ___ (___% of total debits). This identifies where most recorded spending went.


## Reflection

This project helped me practise using Pandas and NumPy to clean transaction data and identify spending patterns. The most challenging part was extracting vendor names from inconsistent transaction descriptions and standardising dates and amounts. I learned how category-wise summaries and z-scores can highlight spending patterns and unusually large transactions. I would improve the project by expanding the vendor keyword dictionary, validating classifications, and adding visualisations.

**AI-assistance disclosure:** AI assistance was used for parts of the coding workflow; I reviewed and adapted the code and should verify the final results against the dataset.
